Modelling

In [ ]:
sampling_interval = '1h'
sequence_len = 18
pred_offset = 6
params = f'{sampling_interval}_{str(sequence_len)}_{str(pred_offset)}'
hb_threshold = 8

import os
os.makedirs(f'model', exist_ok=True)
os.makedirs(f'model/{params}', exist_ok=True)
os.makedirs(f'plots', exist_ok=True)
os.makedirs(f'plots/{params}', exist_ok=True)


import numpy as np
import pandas as pd
import tqdm
import argparse
import os


tqdm.tqdm.pandas()
pd.set_option('display.max_columns', None)

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split


df = pd.read_parquet(f'/home/lkapral/hb/data/mimic_df.parquet')





In [ ]:
df

In [ ]:
icustays = pd.read_csv('data/icustays.csv')

icustays = icustays.rename(columns={"stay_id": "encounterId"})

good_icus = ['Medical Intensive Care Unit (MICU)',
       'Surgical Intensive Care Unit (SICU)',
       'Medical/Surgical Intensive Care Unit (MICU/SICU)',
       'Cardiac Vascular Intensive Care Unit (CVICU)',
       'Coronary Care Unit (CCU)', 'Neuro Intermediate',
       'Trauma SICU (TSICU)', 'Neuro Stepdown',
       'Neuro Surgical Intensive Care Unit (Neuro SICU)']
df = df.merge(icustays[['encounterId', 'first_careunit']], on='encounterId')



In [ ]:
icustays['first_careunit'].value_counts()

In [ ]:
df = df.loc[df['first_careunit'].isin(good_icus), :].reset_index(drop=True)

In [ ]:
df.drop(columns=['first_careunit'], inplace=True)

In [ ]:
df['encounterId'].nunique()

In [ ]:
df['hemoglobin_g/dl'].describe()

In [ ]:
df['hemoglobin_g/dl'].value_counts()

In [ ]:
for col in df.columns:
    print(col, df[col].count())

In [ ]:
df.columns

In [ ]:
len(df['encounterId'].unique())

In [ ]:
df

In [ ]:


df.sort_values(by=['encounterId', 'utcChartTime'], inplace=True)

df = df.reset_index(drop=True)


In [ ]:
df['sex_or_gender'].value_counts()

In [ ]:
df

In [ ]:



# 1. Ensure the DataFrame is sorted by encounter and time so "first" is meaningful
df = df.sort_values(by=['encounterId', 'utcChartTime'])

df['Hb_not_NaN'] = df['hemoglobin_g/dl'].notna().astype(int)
# 2. Create a mask for the first 'Hb_not_NaN' == 1 per group
#    - We look for rows where Hb_not_NaN is 1
#    - AND the cumulative sum of Hb_not_NaN within the group is 1 (meaning it's the first one)
first_occurrence_mask = (df['Hb_not_NaN'] == 1) & (df.groupby('encounterId')['Hb_not_NaN'].cumsum() == 1)

# 3. Set those specific values to 0
df.loc[first_occurrence_mask, 'Hb_not_NaN'] = 0

# Optional: Verify the change
print(f"Modified {first_occurrence_mask.sum()} entries (first measurement set to 0).")



# Define the mapping dictionary
gender_mapping = {'M': 0, 'F': 1}

# Apply the mapping to create a new numeric column
df['sex_or_gender_numeric'] = df['sex_or_gender'].map(gender_mapping)

# Check for any NaN values introduced by unmapped categories (e.g., 'W')
num_missing = df['sex_or_gender_numeric'].isna().sum()
print(f"Number of unmapped 'sex_or_gender' entries: {num_missing}")

# Assign a default value for unmapped categories (e.g., 2 for 'Unknown')
df['sex_or_gender'] = df['sex_or_gender_numeric'].fillna(2).astype(int)

# Drop the original 'sex_or_gender' column if it's no longer needed
df = df.drop(columns=['sex_or_gender_numeric'])

# Update feature_cols to include the new numeric gender column
feature_cols = df.columns.difference(['Hb_not_NaN', 'encounterId', 'utcChartTime'])




In [ ]:
# Count unique encounters before filtering
encounter_count_before = df['encounterId'].nunique()

# Filter to keep only those encounters that have at least one row with Hb_not_NaN == 1
df_filtered = df.groupby('encounterId').filter(lambda group: (group['Hb_not_NaN'] == 1).any())

# Count unique encounters after filtering
encounter_count_after = df_filtered['encounterId'].nunique()

# Calculate how many encounters were removed
removed_encounters = encounter_count_before - encounter_count_after

print(f"Number of removed encounters: {removed_encounters}")


In [ ]:
df['Hb_not_NaN'].value_counts()

In [ ]:
import pandas as pd

def trim_sequences(
    df,
    group_col='encounterId',
    time_col='utcChartTime',
    hb_col='Hb_not_NaN',
    sequence_len=6,
    sampling_interval='6h'
):
    """
    Filters out encounters where there is no row with `hb_col == 1`.
    Then, for each encounter, it trims rows so that:
      - The start time is `first_Hb_time - (sequence_len * sampling_interval)`
      - The end time is `last_Hb_time`
    
    Parameters
    ----------
    df : pd.DataFrame
        The original DataFrame.
    group_col : str
        Name of the column to group by, e.g. 'encounterId'.
    time_col : str
        Name of the datetime column, e.g. 'utcChartTime'.
    hb_col : str
        Name of the column that indicates `Hb_not_NaN`.
    sequence_len : int
        Number of time-steps to look back before the first `hb_col == 1`.
    sampling_interval : str or pd.Timedelta
        The time interval per step (e.g., '6h', '12h', '30T' for minutes, etc.).
        If str, it will be converted via `pd.Timedelta(sampling_interval)`.

    Returns
    -------
    pd.DataFrame
        A new DataFrame with encounters trimmed to the desired window.
    """

    # 1. Filter out encounters that never have hb_col == 1
    df_filtered = df.groupby(group_col).filter(lambda g: (g[hb_col] == 1).any())

    # 2. Sort by group_col and time_col
    df_filtered = df_filtered.sort_values([group_col, time_col])

    # 3. Convert sampling_interval to pd.Timedelta if it's a string
    if isinstance(sampling_interval, str):
        sampling_interval = pd.Timedelta(sampling_interval)

    # 4. Define a function to trim each encounter
    def _trim_single_encounter(group):
        # Find first and last times where hb_col == 1
        first_time = group.loc[group[hb_col] == 1, time_col].min()
        last_time  = group.loc[group[hb_col] == 1, time_col].max()

        # Total look-back time
        total_delta = sequence_len * sampling_interval

        # Calculate start and end times
        t_start = first_time - total_delta
        t_end = last_time

        # Keep rows in [t_start, t_end]
        mask = (group[time_col] >= t_start) & (group[time_col] <= t_end)
        return group.loc[mask]

    # 5. Apply the trimming function
    df_trimmed = df_filtered.groupby(group_col, group_keys=False).apply(_trim_single_encounter)

    # 6. Reset index (optional)
    df_trimmed = df_trimmed.reset_index(drop=True)

    return df_trimmed


In [ ]:
df_trimmed = trim_sequences(
    df,
    group_col='encounterId',       # default
    time_col='utcChartTime',       # default
    hb_col='Hb_not_NaN',           # default
    sequence_len=sequence_len,
    sampling_interval=sampling_interval
)

In [ ]:
df = df_trimmed

In [ ]:
df['encounterId'].nunique()

In [ ]:
df['y'] = 0

df.loc[df['Hb_not_NaN']==1, 'y'] = df.loc[df['Hb_not_NaN']==1, 'hemoglobin_g/dl'].le(hb_threshold).astype(int)

df[['y', 'hemoglobin_g/dl']]

In [ ]:
from sklearn.model_selection import train_test_split

encounter_ids = df['encounterId'].unique()
train_ids, test_ids = train_test_split(encounter_ids, train_size=1, random_state=42)
df_train = df[df['encounterId'].isin(train_ids)].copy()
df_test = df[df['encounterId'].isin(test_ids)].copy()





# Identify feature columns
feature_cols = df.columns.difference(['Hb_not_NaN', 'encounterId', 'utcChartTime', 'y'])
print("Feature Columns:", feature_cols.tolist())

# Identify non-numeric columns within feature_cols
non_numeric_cols = df[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-Numeric Feature Columns:", non_numeric_cols)

In [ ]:
df_test['y'].value_counts()

In [ ]:
df_train['y'].value_counts()

In [ ]:
feature_cols = [col for col in feature_cols if col not in non_numeric_cols]
print("Updated Feature Columns (Numeric Only):", feature_cols)

feature_cols_ids = ['encounterId']+list(feature_cols)

In [ ]:
df_train[feature_cols] = df_train[feature_cols].astype(np.float32)
df_test[feature_cols] = df_test[feature_cols].astype(np.float32)

In [ ]:
# Sort the DataFrame by 'encounterId' and 'utcChartTime'
df_train.sort_values(['encounterId', 'utcChartTime'], inplace=True)
df_test.sort_values(['encounterId', 'utcChartTime'], inplace=True)

# Forward fill within each 'encounterId' group using groupby().ffill()
df_train[feature_cols_ids] = df_train.groupby('encounterId')[feature_cols_ids].ffill()
df_test[feature_cols_ids] = df_test.groupby('encounterId')[feature_cols_ids].ffill()

In [ ]:
# Forward fill within each 'encounterId' group using groupby().ffill()
df_train[feature_cols_ids] = df_train.groupby('encounterId')[feature_cols_ids].bfill()
df_test[feature_cols_ids] = df_test.groupby('encounterId')[feature_cols_ids].bfill()

# Debugging Statements
print("\nAfter Backfill - Training DataFrame:")
df_train.head()
print("Columns:", df_train.columns.tolist())


In [ ]:

# Compute median values for training data
median_values = df_train[feature_cols].median()

# Compute median values for training data
median_values = df_train[feature_cols].median()

# Fill NaNs in training data
df_train[feature_cols] = df_train[feature_cols].fillna(median_values)

# Fill NaNs in test data using training median values
df_test[feature_cols] = df_test[feature_cols].fillna(median_values)

# Verify that there are no NaNs left
assert not df_train[feature_cols].isnull().values.any(), "NaNs remain in df_train"
assert not df_test[feature_cols].isnull().values.any(), "NaNs remain in df_test"


# Debugging Statements
print("\nAfter Median Fill - Training DataFrame:")
print("Columns:", df_train.columns.tolist())

print("\nAfter Median Fill - Test DataFrame:")
print("Columns:", df_test.columns.tolist())


In [ ]:
df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

In [ ]:
time_variables = ['heart_rate', 'respiratory_rate', 'blood_pressure_systolic_mmHg', 'spo2',
       'blood_pressure_mean_mmHg', 'blood_pressure_diastolic_mmHg',
       'combined_vaso',
       'fluids_ml', 'colloids_ml', 'lactate_mmol/l', 'base_excess_mmol/l',
       'hemoglobin_g/dl', 'fibrinogen_mg/dl', 'platelet_count_G/l', 'harnk_ml',
       'drain_sum',  'blood_input']

In [ ]:
def add_lags_and_y_target(
    df,
    group_col='encounterId',
    sort_col='utcChartTime',
    sequence_len=6,
    hb_flag_col='Hb_not_NaN',
    hemoglobin_col='hemoglobin_g/dl',
    target_col='y_target',
    interpolation_method='pchip',   # default
    interpolation_order=3           # used only for spline/poly
):
    """
    For each group (identified by group_col):
      1. Sort rows by sort_col.
      2. Mask real hemoglobin where hb_flag_col != 1 (set to NaN).
      3. Interpolate hemoglobin values; store in target_col.
      4. Create lag columns for each variable in time_variables.
    """
    # 1) Sort rows within each group by the sort_col
    df_sorted = df.sort_values([group_col, sort_col]).copy()

    df_sorted['hemo_helper'] = df_sorted[hemoglobin_col]

    # 2) Where hb_flag_col != 1, set hemoglobin to NaN
    df_sorted.loc[df_sorted[hb_flag_col] != 1, 'hemo_helper'] = np.nan

    # 3) Interpolation with method-specific minimum point logic
    def _interp_hemoglobin(g):
        interpolated_series = g['hemo_helper']

        real_values_count = interpolated_series.notna().sum()

        # Minimum points required for each method
        min_points_for_linear = 2
        min_points_for_pchip = 2
        min_points_for_spline = interpolation_order + 1

        if interpolation_method in ['spline', 'polynomial']:
            min_points_needed = min_points_for_spline
        elif interpolation_method == 'pchip':
            min_points_needed = min_points_for_pchip
        else:  # default for linear and other simple methods
            min_points_needed = min_points_for_linear

        if real_values_count >= min_points_needed:
            # Ideal case: use the chosen method
            try:
                if interpolation_method in ['spline', 'polynomial']:
                    interpolated_series = interpolated_series.interpolate(
                        method=interpolation_method,
                        order=interpolation_order,
                        limit_direction='both'
                    )
                else:
                    # pchip / linear / others (no order kwarg)
                    interpolated_series = interpolated_series.interpolate(
                        method=interpolation_method,
                        limit_direction='both'
                    )
            except ValueError:
                # Fallback to linear
                interpolated_series = interpolated_series.interpolate(
                    method='linear',
                    limit_direction='both'
                )
        elif real_values_count >= 2:
            # Not enough points for spline/pchip but at least 2 -> linear
            interpolated_series = interpolated_series.interpolate(
                method='linear',
                limit_direction='both'
            )

        # Final fallback: ffill + bfill to ensure no NaNs remain
        g['hemo_helper'] = interpolated_series.ffill().bfill()
        return g

    df_interp = df_sorted.groupby(group_col, group_keys=False).apply(_interp_hemoglobin)

    # Copy the interpolated hemoglobin to target_col
    df_interp[target_col] = df_interp['hemo_helper']

    # 4) Create lag columns
    def _create_lags_for_group(subdf):
        lag_dfs = []

        # Use first row as fallback (assuming prior global cleaning)
        last_values = subdf[time_variables].iloc[0]

        for var in time_variables:
            if var not in subdf.columns:
                continue

            shifts = {}
            for i in range(1, sequence_len + 1):
                col = subdf[var].shift(i)
                col = col.fillna(last_values[var])  # fill lags within short encounters
                shifts[f'{var}_lag_{i}'] = col

            lag_dfs.append(pd.DataFrame(shifts, index=subdf.index))

        if lag_dfs:
            all_lags = pd.concat(lag_dfs, axis=1)
            return pd.concat([subdf, all_lags], axis=1)
        else:
            return subdf

    df_lagged = df_interp.groupby(group_col, group_keys=False).apply(_create_lags_for_group)
    df_lagged.drop(columns='hemo_helper', inplace=True)
    df_lagged.reset_index(drop=True, inplace=True)

    return df_lagged


In [ ]:
df_train = add_lags_and_y_target(
    df_train,
    group_col='encounterId',
    sort_col='utcChartTime',
    sequence_len=sequence_len,         
    hb_flag_col='Hb_not_NaN',
    hemoglobin_col='hemoglobin_g/dl',
    target_col='y_target',
    interpolation_method='linear'
)

df_test = add_lags_and_y_target(
    df_test,
    group_col='encounterId',
    sort_col='utcChartTime',
    sequence_len=sequence_len,         
    hb_flag_col='Hb_not_NaN',
    hemoglobin_col='hemoglobin_g/dl',
    target_col='y_target',
    interpolation_method='linear'
)

In [ ]:
# Identify feature columns
feature_cols = df_train.columns.difference(['Hb_not_NaN', 'encounterId', 'utcChartTime', 'y_target'])
print("Feature Columns:", feature_cols.tolist())




In [ ]:
len(feature_cols)

In [ ]:
feature_cols

In [ ]:
# Helper function using the FixedForwardWindowIndexer
# We use min_periods=1 to capture if ANY measurement occurred in the window
def get_future_min_measured(series, mask, window):
    # 1. Create a series that only contains REAL measurements (NaN otherwise)
    #    This prevents the target from seeing forward-filled values.
    real_values = series.where(mask == 1)
    
    # 2. Define the forward looking window
    indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=window)
    
    # 3. Shift by -1 to start looking from t+1 to t+window
    #    If min_periods=1, this returns a value only if there is at least
    #    ONE real measurement in the next 6 hours. Otherwise, it returns NaN.
    return real_values.shift(-1).rolling(window=indexer, min_periods=1).min()

# # --- APPLY TO TRAIN ---
# df_train['future_min_hb'] = df_train.groupby('encounterId').apply(
#     lambda x: get_future_min_measured(
#         x['hemoglobin_g/dl'], 
#         x['Hb_not_NaN'],  # Use the mask to ensure we only look at real measurements
#         pred_offset
#     )
# ).reset_index(level=0, drop=True)

# --- APPLY TO TEST ---
df_test['future_min_hb'] = df_test.groupby('encounterId').apply(
    lambda x: get_future_min_measured(
        x['hemoglobin_g/dl'], 
        x['Hb_not_NaN'], 
        pred_offset
    )
).reset_index(level=0, drop=True)


#df_train = df_train.dropna(subset=['future_min_hb'])
df_test = df_test.dropna(subset=['future_min_hb'])

# 5. Create the binary target
df_train['y_binary'] = 0
df_test['y_binary'] = (df_test['future_min_hb'] < hb_threshold).astype(int)

print(f"Train samples with valid future measurements: {len(df_train)}")
print(f"Test samples with valid future measurements: {len(df_test)}")

df_train = df_train.drop(columns=['future_min_hb'], errors='ignore')
df_test = df_test.drop(columns=['future_min_hb'], errors='ignore')

In [ ]:
import pandas as pd

# Suppose your DataFrame is named df_train
columns_with_nan = df_train.columns[df_train.isna().any()].tolist()
print(columns_with_nan)


In [ ]:
df_train['y_binary'].value_counts()

In [ ]:
df_test['y_binary'].value_counts()/len(df_test)

In [ ]:
df_test['encounterId'].nunique()

In [ ]:
import numpy as np
import pandas as pd
import tqdm
import argparse
import os


tqdm.tqdm.pandas()
pd.set_option('display.max_columns', None)

In [ ]:
import numpy as np
import pandas as pd
import os
import itertools
import json
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib


In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
import joblib




In [ ]:
def select_specific_lags(df: pd.DataFrame, config: dict) -> pd.DataFrame:
    """
    Retains or transforms lag columns for specified time variables according to a configuration dict.

    Parameters
    ----------
    df : pd.DataFrame
        The DataFrame containing original features along with lagged features.
    config : dict
        Mapping of each time-variable column name to a dict with keys:
          - 'lags': list of int
              Lag steps to consider (e.g. [1, 2, 3]).
          - 'mode': str
              One of:
                * 'normal': keep each lag as its own column (var_lag_N)
                * 'diff': compute difference between each lag and the current value (var_lag_N_diff)
                * 'sum': sum all lagged values into a single column (var_lags_sum)

        Example:
            {
                'blood_pressure_systolic_mmHg': {'lags': [1,2,3], 'mode': 'diff'},
                'lactate_mmol/l': {'lags': [1,2,3,6], 'mode': 'sum'},
            }

    Returns
    -------
    pd.DataFrame
        A DataFrame containing the original non-lag columns plus the selected or transformed lag features.
    """
    # Identify all specified variables
    time_vars = list(config.keys())

    # 1. Identify original (non-lag) columns
    non_lag_cols = [
        col for col in df.columns
        if not any(col.startswith(f"{var}_lag_") for var in time_vars)
    ]

    # 2. Initialize output with non-lag columns
    df_out = df[non_lag_cols].copy()

    # 3. Process each variable according to its mode
    for var, params in config.items():
        lags = params.get('lags', [])
        mode = params.get('mode', 'normal').lower()

        # Find existing lag columns for this variable
        existing = [(lag, f"{var}_lag_{lag}") for lag in lags if f"{var}_lag_{lag}" in df.columns]
        if not existing:
            continue

        if mode == 'normal':
            # Keep each lag as-is
            for lag, col in existing:
                df_out[col] = df[col]

        elif mode == 'diff':
            # Difference between lag value and current value
            if var not in df.columns:
                raise KeyError(f"Current value column '{var}' not found for diff mode.")
            for lag, col in existing:
                out_col = f"{var}_lag_{lag}_diff"
                df_out[out_col] = df[col] - df[var]

        elif mode == 'sum':
            # Sum all lag columns into one
            cols_to_sum = [col for _, col in existing]
            out_col = f"{var}_lags_{max(lags)}_sum"
            df_out[out_col] = df[cols_to_sum].sum(axis=1)

        else:
            raise ValueError(f"Unsupported mode '{mode}' for variable '{var}'.")

    return df_out


In [ ]:
lags_config={
  "heart_rate": {
    "lags": [1, 6, 18],
    "mode": "diff"
  },
  "respiratory_rate": {
    "lags": [6, 18],
    "mode": "diff"
  },
  "blood_pressure_systolic_mmHg": {
    "lags": [6],
    "mode": "diff"
  },
  "spo2": {
    "lags": [6],
    "mode": "diff"
  },
  "blood_pressure_mean_mmHg": {
    "lags": [6, 18],
    "mode": "diff"
  },
  "blood_pressure_diastolic_mmHg": {
    "lags": [6],
    "mode": "diff"
  },
  "combined_vaso": {
    "lags": [18],
    "mode": "diff"
  },
  "lactate_mmol/l": {
    "lags": [6, 18],
    "mode": "diff"
  },
  "base_excess_mmol/l": {
    "lags": [6, 18],
    "mode": "diff"
  },
  "hemoglobin_g/dl": {
    "lags": [6, 12, 18],
    "mode": "diff"
  },
  "fibrinogen_mg/dl": {
    "lags": [6, 18],
    "mode": "diff"
  },
  "platelet_count_G/l": {
    "lags": [6, 18],
    "mode": "diff"
  },
  "harnk_ml": {
    "lags": [1, 2, 3, 4, 5, 6],
    "mode": "sum"
  },
  "drain_sum": {
    "lags": [1, 2, 3, 4, 5, 6],
    "mode": "sum"
  },
  "blood_input": {
    "lags": [1, 2, 3, 4, 5, 6],
    "mode": "sum"     
  }, "fluids_ml": {
    "lags": [1, 2, 3, 4, 5, 6],
    "mode": "sum"
  },
  "colloids_ml": {
    "lags": [1, 2, 3, 4, 5, 6],
    "mode": "sum"
  },
}

In [ ]:
sum_config={
"harnk_ml": {
    "lags": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
    "mode": "sum"
  },
  "drain_sum": {
    "lags": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
    "mode": "sum"
  },
  "blood_input": {
    "lags": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
    "mode": "sum"
  },
  "fluids_ml": {
    "lags": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
    "mode": "sum"
  },
  "colloids_ml": {
    "lags": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18],
    "mode": "sum"
  },
}

In [ ]:
df_train_lags = df_train.copy()

In [ ]:
df_test_lags = df_test.copy()

In [ ]:
# Apply to the training set
df_train= select_specific_lags(
    df=df_train_lags,
    config=lags_config
)

# Apply to the test set
df_test = select_specific_lags(
    df=df_test_lags,
    config=lags_config
)

In [ ]:
# Apply to the training set
df_train_18= select_specific_lags(
    df=df_train_lags,
    config=sum_config
)

# Apply to the test set
df_test_18 = select_specific_lags(
    df=df_test_lags,
    config=sum_config
)

In [ ]:
cols_to_add = df_train_18.columns.difference(df_train_lags.columns)


In [ ]:
print("cols_to_add:", cols_to_add.tolist())

In [ ]:
df_train[cols_to_add] = df_train_18[cols_to_add]
df_test[cols_to_add] = df_test_18[cols_to_add]

In [ ]:
# --- Add this as a NEW CELL before Cell 51 ---

# We only want to predict a *transition* into a low state.
# Let's define "not low" as having a recent Hb > 9.5 g/dl.
stable_hb_threshold = hb_threshold 

# This is the most recent Hb value the model can see
most_recent_hb_col = 'hemoglobin_g/dl' 

# # Filter the training and test sets
# # We only want samples where the patient was NOT already in a low state
# df_train_filtered = df_train[df_train[most_recent_hb_col] > stable_hb_threshold].copy()
# df_test_filtered = df_test[df_test[most_recent_hb_col] > stable_hb_threshold].copy()





In [ ]:
feature_cols = df_train.columns.difference(['Hb_not_NaN', 'encounterId', 'utcChartTime', 'y_binary', 'y_target', 'y'])
print("Feature Columns:", feature_cols.tolist())

In [ ]:
# --- NOW, in Cell 51, use these new filtered DataFrames ---
X_train = df_train[feature_cols].values 
#X_train = df_train_filtered[feature_cols].values # <-- NEW

# --- In Cell 52, use these new filtered DataFrames ---
y_train = df_train['y_binary'].values  
#y_train = df_train_filtered['y_binary'].values # <-- NEW

# And so on for X_test and y_test
X_test = df_test[feature_cols].values
y_test = df_test['y_binary'].values


#NON NAN EXCLUSION

In [ ]:



# df_train = df_train.loc[df_train['Hb_not_NaN']==1,:].reset_index()
# df_test = df_test.loc[df_test['Hb_not_NaN']==1,:].reset_index()


In [ ]:
len(feature_cols)

## y_test_binary = np.load(f'model/{params}/y_test_binary.npy')  

In [ ]:
X_train = df_train[feature_cols].values
X_test = df_test[feature_cols].values

In [ ]:
y_train = df_train['y_binary'].values
y_test = df_test['y_binary'].values

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline


In [ ]:
df_train.columns

In [ ]:
X_train.shape

In [ ]:
feature_cols

In [ ]:
import numpy as np
import pandas as pd

def compute_transition_weights(
    enc_ids, 
    y, 
    balance_classes=True, 
    switch_boost_factor=5.0, 
    transition_type='0to1'
):
    """
    Computes sample weights with class balancing and transition boosting.

    1. Applies base weights to balance classes (Req 1).
    2. Applies a 'switch_boost_factor' to transitions (Req 2).
    3. Allows specifying '0to1', '1to0', or 'both' transitions (Req 3).
    """
    
    # --- 1. Set Base Weights for Class Balancing (Req 1) ---
    if balance_classes:
        n_total = len(y)
        n_class_0 = np.sum(y == 0)
        n_class_1 = np.sum(y == 1)
        
        # Weight for each class: n_total / (n_classes * n_samples_in_class)
        # This equalizes the total weight contribution of each class.
        weight_0 = n_total / (2.0 * n_class_0)
        weight_1 = n_total / (2.0 * n_class_1)
        
        # Create the base sample_weight array
        sw = np.where(y == 1, weight_1, weight_0).astype(float)
        
        print(f"Base weights set for class balance: Class 0={weight_0:.2f}, Class 1={weight_1:.2f}")
        
    else:
        # Default to all ones if not balancing
        sw = np.ones_like(y, dtype=float)

        
    # --- 2. Find Transitions and Apply Boost (Req 2 & 3) ---
    
    # Create a DataFrame to easily group by encounter
    # This df has a default integer index [0, 1, 2...]
    df_sw = pd.DataFrame({'enc': enc_ids, 'y': y})
    
    # This will store the *integer positions* [0, 1, 2...] of the
    # samples that need to be boosted.
    indices_to_boost = [] 

    for eid, grp in df_sw.groupby('enc', sort=False):
        arr = grp['y'].values
        
        # [IMPROVEMENT] Find *all* flips, not just the first one
        
        if transition_type in ('0to1', 'both'):
            # Find indices where the transition 0->1 *ends* (at arr[i])
            flips_0to1 = np.where((arr[:-1] == 0) & (arr[1:] == 1))[0] + 1
            if flips_0to1.size > 0:
                # Get the original integer positions from the group's index
                indices_to_boost.extend(grp.index[flips_0to1])
        
        if transition_type in ('1to0', 'both'):
            # Find indices where the transition 1->0 *ends* (at arr[i])
            flips_1to0 = np.where((arr[:-1] == 1) & (arr[1:] == 0))[0] + 1
            if flips_1to0.size > 0:
                # Get the original integer positions from the group's index
                indices_to_boost.extend(grp.index[flips_1to0])

    if indices_to_boost:
        # Remove duplicates if 'both' was selected and some indices overlapped
        indices_to_boost = np.unique(indices_to_boost)
        
        # Apply the boost factor as a *multiplier* on the base weight
        sw[indices_to_boost] = sw[indices_to_boost] * switch_boost_factor
        print(f"Applied {switch_boost_factor}x boost to {len(indices_to_boost)} transition samples.")
    else:
        print(f"No transitions of type '{transition_type}' found to boost.")

    return sw

In [ ]:
train_weights = compute_transition_weights(
    df_train['encounterId'].values, 
    y_train,
    balance_classes=True,
    switch_boost_factor=10.0, 
    transition_type='0to1'
)

In [ ]:
tw = pd.DataFrame(train_weights, columns=['weights'])

In [ ]:
df_train['encounterId'].unique

In [ ]:
df_test.columns

In [ ]:
tw.loc[tw['weights']==1000,:]

In [ ]:
tw.loc[tw['weights']==1000,:]

In [ ]:
len(df_test['encounterId'].unique())

In [ ]:
import pandas as pd

mimic_ids = df_test['encounterId'].unique()
# Convert to Series to use pandas' built-in CSV saver
pd.Series(mimic_ids).to_csv('mimic_ids.csv', index=False, header=['encounterId'])

In [ ]:
df_test['y_binary'].value_counts()

In [ ]:
df_test.columns

In [ ]:
# base_dir = Path("xgb_models_best")
# base_dir.mkdir(exist_ok=True)

# param_file = base_dir / "best_params.json"
# model_file = base_dir / "best_pipeline.pkl"


# Define the base directory and load the saved pipeline
# base_dir = 'xgb_models_best'
# model_path = os.path.join(base_dir, 'best_pipeline.pkl')
# best_pipeline = joblib.load(model_path)


base_dir = 'xgb_models_best'
model_path = os.path.join(base_dir, 'calibrated_pipeline.pkl')
best_pipeline = joblib.load(model_path)

# Predict on the test set

y_pred = best_pipeline.predict(X_test)
y_pred_proba = best_pipeline.predict_proba(X_test)[:, 1]

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
auc_score = roc_auc_score(y_test, y_pred_proba)

# Print metrics
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test AUC: {auc_score:.4f}")

# Save the metrics to a JSON file
metrics = {
    'Accuracy': accuracy,
    'Precision': precision,
    'Recall': recall,
    'F1 Score': f1,
    'AUC': auc_score
}

metrics_path = os.path.join(base_dir, 'test_metrics.json')
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=4)

In [ ]:
from sklearn.metrics import confusion_matrix

# After you have y_test and y_pred defined:
cm = confusion_matrix(y_test, y_pred)
print("Confusion matrix:\n", cm)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Ensure base_dir is defined and exists
base_dir = 'xgb_models_best'
os.makedirs(base_dir, exist_ok=True)

# Number of bootstrap samples
n_bootstraps = 100
rng = np.random.RandomState(seed=42)  # For reproducibility

# Arrays to store bootstrapped FPR and TPR
bootstrapped_fpr = []
bootstrapped_tpr = []

# Convert y_test and X_test to numpy arrays if they aren't already
y_test_np = np.array(y_test)
X_test_np = np.array(X_test)
y_pred_proba_np = np.array(y_pred_proba)

for i in range(n_bootstraps):
    # Sample with replacement from the test set
    indices = rng.randint(0, len(X_test_np), len(X_test_np))
    
    # Ensure both classes are present in the bootstrap sample
    if len(np.unique(y_test_np[indices])) < 2:
        continue
    
    # Extract the sampled true labels and predicted probabilities
    y_test_sample = y_test_np[indices]
    y_pred_proba_sample = y_pred_proba_np[indices]
    
    # Compute ROC curve
    fpr, tpr, _ = roc_curve(y_test_sample, y_pred_proba_sample)
    bootstrapped_fpr.append(fpr)
    bootstrapped_tpr.append(tpr)

# Define mean FPR for interpolation
mean_fpr = np.linspace(0, 1, 100)

# Interpolate TPRs at the mean FPR points
interpolated_tprs = []
for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr):
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0  # Ensure the curve starts at (0,0)
    interpolated_tprs.append(interp_tpr)

# Convert list to array for percentile calculation
interpolated_tprs = np.array(interpolated_tprs)

# Compute mean and confidence intervals
mean_tpr = np.mean(interpolated_tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)
tpr_lower = np.percentile(interpolated_tprs, 5, axis=0)
tpr_upper = np.percentile(interpolated_tprs, 95, axis=0)

auc = auc(fpr, tpr)

mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.array([np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr)])



# Compute the lower and upper bound for tpr
tpr_lower = np.percentile(mean_tpr, 5, axis=0)
tpr_upper = np.percentile(mean_tpr, 95, axis=0)

# Specificity is 1 - FPR
specificity =  1 - fpr

# Set the style and context with seaborn
sns.set_style("ticks")
sns.set_context("notebook")



# Specificity is 1 - FPR
specificity =  1 - fpr





# Plotting
plt.figure(figsize=(8, 6))

plt.plot([1, 0], [0, 1], 'k--')  # Invert x-axis
plt.plot(specificity, tpr, label=f'(AUC = {auc:.2f})')
plt.fill_between(1-mean_fpr, tpr_lower, tpr_upper, alpha=0.2)

plt.xlim([1, 0])  # Invert x-axis limits
plt.ylim([0, 1])
#plt.xticks([1,0.8,0.6,0.4,0.2,0])  # Set x-axis tick locations
#plt.xticks([1,0.8,0.6,0.4,0.2,0])  # Set x-axis tick labels

# Labels and Title
plt.xlabel('Specificity')
plt.ylabel('Sensitivity')
plt.title('ROC curve for hemoglobin prediction', fontsize=16)
# Legend
plt.legend(loc="lower right")

# Ensure equal aspect ratio
plt.gca().set_aspect('equal')

# Tight layout for better spacing
plt.tight_layout()

# Save the ROC curve
roc_curve_path = os.path.join(base_dir, 'roc_curve_bootstrapped_MIMIC.png')
plt.savefig(roc_curve_path)
plt.show()
plt.close()
print(f"Bootstrapped ROC curve saved to {roc_curve_path}")

In [ ]:
df_train

In [ ]:
X_train.shape

In [ ]:
enc_ids = df_test['encounterId'].values

In [ ]:
def encounter_grouped_auc(encounter_ids, y_true, y_proba):
    """
    Compute the mean AUC across encounters, giving each encounter equal weight.

    Parameters
    ----------
    encounter_ids : array-like or pd.Series
        Encounter identifier for each sample (same order as y_true / y_proba).
    y_true : array-like or pd.Series
        True binary labels (0/1).
    y_proba : array-like or pd.Series
        Predicted positive-class probabilities.

    Returns
    -------
    float
        Macro‐average AUC over all encounters that have at least one 0 and one 1.
    pd.Series
        The per‐encounter AUC values (index = encounterId).
    """
    df = pd.DataFrame({
        'enc': encounter_ids,
        'y': y_true,
        'p': y_proba
    })
    aucs = {}
    for eid, grp in df.groupby('enc'):
        if grp['y'].nunique() < 2:
            # skip encounters without both classes
            continue
        aucs[eid] = roc_auc_score(grp['y'], grp['p'])
    if not aucs:
        raise ValueError("No encounter has both classes present; cannot compute grouped AUC.")
    per_encounter = pd.Series(aucs)
    return per_encounter.mean(), per_encounter



mean_grouped_auc, per_patient_auc = encounter_grouped_auc(
    encounter_ids=enc_ids,
    y_true=y_test,
    y_proba=y_pred_proba
)

print(f"Macro‐average AUC across encounters: {mean_grouped_auc:.4f}")

# If you want to inspect a few:
print(per_patient_auc.head())


In [ ]:
y_pred

In [ ]:
from sklearn.metrics import f1_score



best_thresh = 0.0
best_f1     = 0.0

for t in np.linspace(0.0, 1.0, 101):
    y_pred_t = (y_pred_proba >= t).astype(int)
    f1 = f1_score(y_test, y_pred_t)
    if f1 > best_f1:
        best_f1     = f1
        best_thresh = t

print("Best threshold for F1:", best_thresh)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import os  # make sure base_dir is defined elsewhere

def plot_encounter_timeseries(
    df,
    encounter_id,
    time_col='utcChartTime',
    hemo_col='hemoglobin_g/dl',
    y_binary_col='y_binary',
    y_target_col='y_target',
    prediction_col='y_pred_proba',
    threshold_line=9,
    path=None,
    offset=0,
    best_thresh=0.5,

):
    # 1) Filter to this encounter
    df_enc = df[df['encounterId'] == encounter_id].copy()
    if df_enc.empty:
        print(f"No rows found for encounterId={encounter_id}.")
        return

    # 2) Ensure time_col is datetime, then sort
    df_enc[time_col] = pd.to_datetime(df_enc[time_col])
    df_enc = df_enc.sort_values(by=time_col)

    # 3) Compute binary prediction
    df_enc['y_pred'] = (df_enc[prediction_col] >= best_thresh).astype(int)

    # 4) Masks for TP, TN, FP, FN
    y_true = df_enc[y_binary_col]
    y_pred = df_enc['y_pred']
    mask_tp = (y_true == 1) & (y_pred == 1)
    mask_tn = (y_true == 0) & (y_pred == 0)
    mask_fp = (y_true == 0) & (y_pred == 1)
    mask_fn = (y_true == 1) & (y_pred == 0)

    # 5) Create figure
    plt.figure(figsize=(10, 6))

    # Plot measured hemoglobin if available
    if hemo_col in df_enc.columns:
        plt.plot(
            df_enc[time_col],
            df_enc[y_target_col],
            label='Hemoglobin',
            marker='x',
            color='tab:blue'
        )

    # 6) Scatter y_target by confusion category
    plt.scatter(
        df_enc.loc[mask_tp, time_col],
        df_enc.loc[mask_tp, y_target_col],
        color='green', marker='o', label='TP (target hemoglobin)'
    )
    plt.scatter(
        df_enc.loc[mask_tn, time_col],
        df_enc.loc[mask_tn, y_target_col],
        color='blue', marker='o', label='TN (target hemoglobin)'
    )
    plt.scatter(
        df_enc.loc[mask_fp, time_col],
        df_enc.loc[mask_fp, y_target_col],
        color='red', marker='o', label='FP (target hemoglobin)'
    )
    plt.scatter(
        df_enc.loc[mask_fn, time_col],
        df_enc.loc[mask_fn, y_target_col],
        color='orange', marker='o', label='FN (target hemoglobin)'
    )

    # 7) Horizontal threshold line
    plt.axhline(
        y=threshold_line,
        color='red',
        linestyle='--',
    )
    plt.text(
        df_enc.iloc[0][time_col],
        threshold_line,
        f'Hb = {threshold_line} g/dl',
        horizontalalignment='left',
        verticalalignment='bottom'
    )

    # 8) Format x-axis: show YYYY-MM-DD HH:MM
    ax = plt.gca()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())  # automatic tick spacing
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d, %H:%M'))
    plt.xticks(rotation=45, ha='right')

    # 9) Labels, legend, save
    plt.title(f"Hb-Prediction for encounter: {encounter_id}")
    plt.xlabel("Time")
    plt.ylabel("Hemoglobin [g/dl]")
    plt.legend()
    plt.tight_layout()

    encounter_path = os.path.join(base_dir, f'{encounter_id}_hb_prediction.png')
    plt.savefig(encounter_path, bbox_inches='tight')
    plt.show()

    print(f"Hb-Prediction saved for {encounter_id} to {encounter_path}")


In [ ]:
df_test['y_pred_proba'] = y_pred_proba

In [ ]:
df_test

In [ ]:
encounters_with_1 = df_test.loc[df_test['y_pred_proba'] < 0.5, 'encounterId'].unique()
print(encounters_with_1)

In [ ]:
import random

In [ ]:

plot_encounter_timeseries(
    df=df_test,
    encounter_id=random.choice(encounters_with_1),
    time_col='utcChartTime',
    hemo_col='hemoglobin_g/dl',
    y_target_col='y_target',
    y_binary_col='y_binary',       # The actual 0/1 label
    prediction_col='y_pred_proba', # The predicted probability column
    threshold_line=hb_threshold,
    path = base_dir,
    offset=0,
    best_thresh=best_thresh
)


In [ ]:
df_test['y_target'].describe()

In [ ]:
df_test['hemoglobin_g/dl'].describe()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

def critical_transition_performance(
    *,
    encounter_ids,
    y_true,
    y_pred=None,
    y_proba=None,
    transition='0to1',  # '0to1', '1to0', or 'both'
    return_df=False
):
    """
    Computes performance at the timepoints where each encounter's label flips.

    Parameters
    ----------
    encounter_ids : array-like or pd.Series
        Encounter identifier for each sample in the same order as y_true.
    y_true : array-like or pd.Series
        True binary labels (0/1).
    y_pred : array-like or pd.Series, optional
        Predicted binary labels (0/1). Required if you want 'accuracy'.
    y_proba : array-like or pd.Series, optional
        Predicted probabilities for the positive class. Required if you want 'auc'.
    transition : str, default '0to1'
        Which flip to capture:
          - '0to1': first transition where label goes from 0 to 1
          - '1to0': first transition where label goes from 1 to 0
          - 'both' : both transitions, metrics returned for each
    return_df : bool, default False
        If True, returns DataFrame(s) of critical samples.

    Returns
    -------
    metrics : dict
        Keys include transition-specific metrics:
          - e.g. 'accuracy_0to1', 'auc_0to1', 'accuracy_1to0', 'auc_1to0'
    critical_df : dict or pd.DataFrame (if return_df=True)
        DataFrame of critical samples per transition type.
        If transition='both', returns dict with keys '0to1' and '1to0'.
    """
    # build base DataFrame
    df = pd.DataFrame({'encounterID': encounter_ids, 'y_true': y_true})
    if y_pred is not None:
        df['y_pred'] = y_pred
    if y_proba is not None:
        df['y_proba'] = y_proba

    # helper to collect flips
    def _find_flips(values, from_val, to_val):
        return np.where((values[:-1] == from_val) & (values[1:] == to_val))[0]

    results = {}
    dfs = {}

    def _process_flip(from_val, to_val, label):
        rows = []
        for eid, grp in df.groupby('encounterID', sort=False):
            y = grp['y_true'].values
            idxs = _find_flips(y, from_val, to_val)
            if len(idxs) == 0:
                continue
            pos = idxs[0] + 1
            rows.append(grp.iloc[pos])
        if not rows:
            return None, None
        crit_df = pd.DataFrame(rows).reset_index(drop=True)
        m = {}
        if y_pred is not None:
            m[f'accuracy_{label}'] = accuracy_score(crit_df['y_true'], crit_df['y_pred'])
        if y_proba is not None:
            m[f'auc_{label}'] = roc_auc_score(crit_df['y_true'], crit_df['y_proba'])
        return m, crit_df

    if transition in ('0to1', 'both'):
        m01, df01 = _process_flip(0, 1, '0to1')
        if m01:
            results.update(m01)
            dfs['0to1'] = df01
    if transition in ('1to0', 'both'):
        m10, df10 = _process_flip(1, 0, '1to0')
        if m10:
            results.update(m10)
            dfs['1to0'] = df10

    if not results:
        raise ValueError(f"No '{transition}' transitions found in any encounter.")

    if return_df:
        # return both metrics and DataFrame(s)
        if transition == 'both':
            return results, dfs
        return results, dfs.get(transition)

    return results


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

def critical_transition_performance(
    *,
    encounter_ids,
    y_true,
    y_pred=None,
    y_proba=None,
    transition='0to1',  # '0to1', '1to0', or 'both'
    return_df=False
):
    """
    Computes performance at the timepoints where each encounter's label flips.

    Parameters
    ----------
    encounter_ids : array-like or pd.Series
        Encounter identifier for each sample in the same order as y_true.
    y_true : array-like or pd.Series
        True binary labels (0/1).
    y_pred : array-like or pd.Series, optional
        Predicted binary labels (0/1). Required if you want 'accuracy'.
    y_proba : array-like or pd.Series, optional
        Predicted probabilities for the positive class. Required if you want 'auc'.
    transition : str, default '0to1'
        Which flip(s) to capture:
          - '0to1': first transition where label goes from 0 to 1
          - '1to0': first transition where label goes from 1 to 0
          - 'both' : compute both and combine results
    return_df : bool, default False
        If True, returns DataFrame(s) of critical samples.

    Returns
    -------
    metrics : dict
        Keys include:
          - 'accuracy_0to1', 'auc_0to1' (if 0→1 computed)
          - 'accuracy_1to0', 'auc_1to0' (if 1→0 computed)
    critical_df : pd.DataFrame (only if `return_df=True`)
        If transition='both', this is a combined DataFrame with a 'transition' column.
        Otherwise it's the DataFrame for the specified flip.
    """
    # Prepare DataFrame
    df = pd.DataFrame({'encounterID': encounter_ids, 'y_true': y_true})
    if y_pred is not None:
        df['y_pred'] = y_pred
    if y_proba is not None:
        df['y_proba'] = y_proba

    def find_flips(values, from_val, to_val):
        return np.where((values[:-1] == from_val) & (values[1:] == to_val))[0]

    def process_flip(from_val, to_val, label):
        rows = []
        for eid, grp in df.groupby('encounterID', sort=False):
            arr = grp['y_true'].values
            idxs = find_flips(arr, from_val, to_val)
            if idxs.size == 0:
                continue
            pos = idxs[0] + 1
            row = grp.iloc[pos].to_dict()
            row['transition'] = label
            rows.append(row)
        if not rows:
            return None, None
        crit_df = pd.DataFrame(rows)
        m = {}
        # Accuracy
        if y_pred is not None:
            m[f'accuracy_{label}'] = accuracy_score(crit_df['y_true'], crit_df['y_pred'])
        # AUC (only if both classes present)
        if y_proba is not None:
            if crit_df['y_true'].nunique() == 2:
                m[f'auc_{label}'] = roc_auc_score(crit_df['y_true'], crit_df['y_proba'])
            else:
                m[f'auc_{label}'] = np.nan
        return m, crit_df

    results = {}
    df_list = []

    # Run for requested transitions
    if transition in ('0to1', 'both'):
        m01, df01 = process_flip(0, 1, '0to1')
        if m01:
            results.update(m01)
            df_list.append(df01)
    if transition in ('1to0', 'both'):
        m10, df10 = process_flip(1, 0, '1to0')
        if m10:
            results.update(m10)
            df_list.append(df10)

    if not results:
        raise ValueError(f"No '{transition}' transitions found.")

    # Combine DataFrames if needed
    combined_df = None
    if return_df:
        if transition == 'both':
            combined_df = pd.concat(df_list, ignore_index=True)
        else:
            combined_df = df_list[0]

    if return_df:
        return results, combined_df
    return results


In [ ]:
metrics, crit_samples = critical_transition_performance(
    encounter_ids=enc_ids,
    y_true=y_test,
    y_pred=y_pred,
    y_proba=y_pred_proba,
    transition='both',
    return_df=True
)

print("0→1 accuracy:", metrics['accuracy_0to1'])
print("1→0 AUC:",      metrics['auc_1to0'])
print(crit_samples.head())


In [ ]:
metrics

In [ ]:
import re
sns.set_theme(style="whitegrid", palette="pastel")
# 1 lag = 1 hour
LAG_HOURS = 1

# --- Base (publication-ready) display names + units ---
BASE_LABELS = {
    "age": ("Age", "years"),
    "sex_or_gender": ("Sex / gender", None),

    "heart_rate": ("Heart rate", "1/min"),
    "respiratory_rate": ("Resp. rate", "1/min"),
    "spo2": ("Oxygen saturation", "%"),

    "blood_pressure_systolic_mmHg": ("Systolic blood pressure", "mmHg"),
    "blood_pressure_diastolic_mmHg": ("Diastolic blood pressure", "mmHg"),
    "blood_pressure_mean_mmHg": ("Mean arterial pressure (MAP)", "mmHg"),

    "lactate_mmol/l": ("Blood lactate", "mmol/L"),
    "base_excess_mmol/l": ("Base excess", "mmol/L"),

    "hemoglobin_g/dl": ("Hemoglobin", "g/dL"),
    "platelet_count_G/l": ("Platelet count", "10⁹/L"),
    "fibrinogen_mg/dl": ("Fibrinogen", "mg/dL"),

    "combined_vaso": ("Vasopressor", "dose"),


    "fluids_ml": ("IV fluids", "mL"),
    "colloids_ml": ("Colloids", "mL"),
    "blood_input": ("Blood products", "mL"),

    "harnk_ml": ("Urine output", "mL"),
    "drain_sum": ("Drain output", "mL"),

    # meta / non-physiological
    "y": ("Outcome", None),
    "y_target": ("Target outcome", None),
    "y_binary": ("Binary outcome", None),
    "Hb_not_NaN": ("Hemoglobin available", None),
    "utcChartTime": ("Chart time (UTC)", None),
    "encounterId": ("Encounter ID", None),
    "y_pred_proba": ("Predicted probability", None),
}


def _fmt_base(name: str) -> str:
    label, unit = BASE_LABELS.get(name, (name, None))
    return f"{label} ({unit})" if unit else label

def scientific_feature_name(feature: str, lag_hours: int = 1) -> str:
    # cumulative sums (e.g. *_lags_6_sum)
    m = re.match(r"^(.*)_lags_(\d+)_sum$", feature)
    if m:
        base, k = m.groups()
        return f"Sum {_fmt_base(base)} over {int(k) * lag_hours} h"

    # temporal differences (e.g. *_lag_6_diff)
    m = re.match(r"^(.*)_lag_(\d+)_diff$", feature)
    if m:
        base, k = m.groups()
        return f"Δ {_fmt_base(base)} over {int(k) * lag_hours} h"

    return _fmt_base(feature)

# --- Build the mapping for your SHAP plot ---
scientific_name_map = {f: scientific_feature_name(f, LAG_HOURS) for f in feature_cols}

# Use mapped names in SHAP summary plot
feature_names_scientific = [scientific_name_map.get(f, f) for f in feature_cols]


import shap 
inner_pipeline = best_pipeline.calibrated_classifiers_[0].estimator 
#inner_pipeline = best_pipeline 
# 1) Prepare the data that goes into the classifier 
# We skip SMOTE at inference time. So we only apply the scaler: 
X_test_scaled = inner_pipeline.named_steps['scaler'].transform(df_test[feature_cols])
# 2) Create a SHAP explainer for the GradientBoostingClassifier 
gbm = inner_pipeline.named_steps['classifier'] 
# The trained GBM inside the pipeline 
explainer = shap.TreeExplainer(gbm)
# Optional but recommended: pass a DataFrame so names stay aligned
import pandas as pd
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names_scientific)

shap_values = explainer.shap_values(X_test_scaled_df)


plt.figure(figsize=(12, 12))
shap.summary_plot(shap_values, X_test_scaled_df, show=False)
plt.title("SHAP feature importance")
plt.tight_layout()
plt.savefig('SHAP_MIMIC.png', bbox_inches="tight")
plt.show()
plt.close()


In [ ]:
# -----------------------------
# 2) “Rolling last label”
#    – for each test row i, predict y_test[i-1]
#      (shifted one step forward)
#    – the first row has no previous label, so default to majority class
# -----------------------------
y_pred_rolling = np.array(df_test['hemoglobin_g/dl']<8).astype(float)
acc_roll = accuracy_score(y_test, y_pred_rolling)
try:
    auc_roll = roc_auc_score(y_test, y_pred_rolling)
except ValueError:
    auc_roll = "undefined (only one class predicted)"

print(f"[ROLLING] accuracy = {acc_roll:.3f}, AUC = {auc_roll}")

In [ ]:
y_pred_rolling

In [ ]:
import numpy as np
import os

# 1. Define Output Directory
output_dir = 'model_outputs'
os.makedirs(output_dir, exist_ok=True)

# 2. Extract Baseline Feature
baseline_values = df_test['hemoglobin_g/dl'].values

# 3. Save Arrays
np.savez(
    os.path.join(output_dir, 'mimic_results.npz'),
    y_true=y_test,
    y_pred_proba=y_pred_proba,
    baseline_hb=baseline_values
)

print(f"MIMIC data saved to {os.path.join(output_dir, 'mimic_results.npz')}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
import numpy as np

# ---------------------------------------------------------
# CONFIGURATION
# ---------------------------------------------------------
n_bins = 10
bin_strategy = 'uniform' # 'uniform' for equal width, 'quantile' for equal sample size

# ---------------------------------------------------------
# CALCULATION
# ---------------------------------------------------------
# Calculate the calibration curve points
prob_true, prob_pred = calibration_curve(y_test, y_pred_proba, n_bins=n_bins, strategy=bin_strategy)

# Calculate Brier Score (Lower is better, 0=Perfect, 0.25=Random Guess for balanced)
brier_score = brier_score_loss(y_test, y_pred_proba)

# ---------------------------------------------------------
# PLOTTING
# ---------------------------------------------------------
# Set a clean professional style
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.1)

# Create figure with 2 subplots (Ratio 3:1)
fig, (ax1, ax2) = plt.subplots(
    nrows=2, 
    ncols=1, 
    figsize=(8, 10), 
    sharex=True, 
    gridspec_kw={'height_ratios': [3, 1]}
)

# --- TOP PLOT: RELIABILITY DIAGRAM ---
# Reference line (Perfect Calibration)
ax1.plot([0, 1], [0, 1], "k:", label="Perfectly Calibrated", alpha=0.6)

# The Model's Curve
ax1.plot(prob_pred, prob_true, "s-", color="#2b8cbe", linewidth=2, markersize=6, label="Model Performance")

# Formatting
ax1.set_ylabel("Observed Frequency (Fraction of Positives)", fontsize=12)
ax1.set_title("Calibration Curve (Reliability Diagram)", fontsize=14, weight='bold')
ax1.legend(loc="lower right", fontsize=11)
ax1.set_ylim([0, 1.])

# Add Brier Score Text Box
textstr = f'Brier Score: {brier_score:.3f}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='lightgrey')
ax1.text(0.05, 0.92, textstr, transform=ax1.transAxes, fontsize=12, verticalalignment='top', bbox=props)


# --- BOTTOM PLOT: FREQUENCY HISTOGRAM ---
# Shows how many predictions fall into each probability bin
sns.histplot(
    x=y_pred_proba, 
    bins=n_bins, 
    binrange=(0,1), 
    ax=ax2, 
    color="#2b8cbe", 
    alpha=0.6, 
    edgecolor="white"
)

# Add counts on top of bars (optional, for explicit "numbers")
# Get the counts and edges from the histogram
counts, edges = np.histogram(y_pred_proba, bins=n_bins, range=(0,1))
centers = 0.5 * (edges[1:] + edges[:-1])
for c, p in zip(centers, counts):
    if p > 0: # Only annotate non-empty bins
        ax2.annotate(str(p), xy=(c, p), xytext=(0, 2), textcoords="offset points", ha='center', fontsize=9)

ax2.set_xlabel("Predicted Probability", fontsize=12)
ax2.set_ylabel("Count", fontsize=12)
ax2.set_yscale('log') # Log scale helps if you have huge imbalance (many 0s, few 1s)
ax2.set_title("Distribution of Predictions", fontsize=11, style='italic')


plt.xlim([0,1])
# Final adjustments
plt.tight_layout()
plt.subplots_adjust(hspace=0.1) # Reduce space between plots

# Save
save_path = 'plots/calibration_curve_beautiful.png'
plt.savefig(save_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to {save_path}")
plt.show()

In [ ]:
import numpy as np
import os

# Create folder if needed
os.makedirs('model_outputs', exist_ok=True)

# Save with a specific name for calibration
np.savez(
    'model_outputs/calibration_external.npz', 
    y_true=y_test,       # Ensure this is your external y_test
    y_pred_proba=y_pred_proba # Ensure this is your external prediction
)

print("External calibration data saved to model_outputs/calibration_external.npz")

In [ ]:
df_test

In [ ]:
df_test['y_binary'].value_counts()/len(df_test)

In [ ]:
df['combined_vaso'].describe()

In [ ]:
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1. SETUP: LOAD DATA
# ---------------------------------------------------------
print("Loading external dataset (MIMIC)...")

if 'df_test' not in locals():
    raise ValueError("Variable 'df_test' not found. Please ensure external data is loaded.")

df = df_test.copy()
df['utcChartTime'] = pd.to_datetime(df['utcChartTime'])

# --- ICU MAPPING ---
if 'first_careunit' not in df.columns:
    try:
        icustays = pd.read_csv('data/icustays.csv')
        if 'stay_id' in icustays.columns:
            icustays = icustays.rename(columns={'stay_id': 'encounterId'})
        df = df.merge(icustays[['encounterId', 'first_careunit']], on='encounterId', how='left')
    except FileNotFoundError:
        print("Warning: 'data/icustays.csv' not found. ICU analysis will be skipped.")
        df['first_careunit'] = 'Unknown'

# ---------------------------------------------------------
# 2. DEFINITIONS (Same as MUW for alignment)
# ---------------------------------------------------------
volumes = ['fluids_ml', 'blood_input', 'colloids_ml', 'harnk_ml', 'drain_sum']
drugs = ['combined_vaso']
vitals = ['heart_rate', 'respiratory_rate', 'spo2', 'blood_pressure_systolic_mmHg', 
          'blood_pressure_diastolic_mmHg', 'blood_pressure_mean_mmHg']
labs = ['hemoglobin_g/dl', 'lactate_mmol/l', 'fibrinogen_mg/dl', 'platelet_count_G/l', 'base_excess_mmol/l']

# ---------------------------------------------------------
# 3. DATA CLEANING (Aligned with MUW)
# ---------------------------------------------------------
print("Cleaning data...")

# Vitals/Labs: Replace 0 with NaN (clinically implausible)
vitals_labs_to_clean = [c for c in (vitals + labs) if c in df.columns]
df[vitals_labs_to_clean] = df[vitals_labs_to_clean].replace(0, np.nan)

# Medications: Replace 0 with NaN (0 = not administered)
drugs_to_clean = [c for c in drugs if c in df.columns]
df[drugs_to_clean] = df[drugs_to_clean].replace(0, np.nan)

# ---------------------------------------------------------
# 4. AGGREGATION - PATIENT LEVEL
# ---------------------------------------------------------
print("Aggregating clinical features to patient level...")

# --- A. VOLUMES: Sum per Day -> Statistics of Daily Sums ---
vol_cols = [c for c in volumes if c in df.columns]
daily_vol_stats = pd.DataFrame()

if vol_cols:
    daily_sums = df.set_index('utcChartTime').groupby(
        ['encounterId', pd.Grouper(freq='1D')]
    )[vol_cols].sum(min_count=1)
    
    daily_vol_stats = daily_sums.groupby('encounterId').agg(['mean', 'median', 'std', 'count'])
    daily_vol_stats.columns = [f"{col[0]}_{col[1]}_daily" for col in daily_vol_stats.columns]

# --- B. MEDICATIONS: Statistics of Administered Doses Only ---
med_stats = pd.DataFrame()
if drugs_to_clean:
    med_stats = df.groupby('encounterId')[drugs_to_clean].agg(['mean', 'median', 'std', 'count'])
    med_stats.columns = [f"{col[0]}_{col[1]}" for col in med_stats.columns]

# --- C. VITALS & LABS ---
mean_vars = [c for c in (vitals + labs) if c in df.columns]
agg_dict = {col: ['mean', 'median', 'std'] for col in mean_vars}

# Add demographics
for c in ['age', 'sex_or_gender', 'first_careunit']:
    if c in df.columns:
        agg_dict[c] = 'first'

patient_vitals_labs = df.groupby('encounterId').agg(agg_dict)

# Flatten column names
new_cols = []
for col in patient_vitals_labs.columns:
    if isinstance(col, tuple):
        if col[1] == 'first':
            new_cols.append(col[0])
        else:
            new_cols.append(f"{col[0]}_{col[1]}")
    else:
        new_cols.append(col)
patient_vitals_labs.columns = new_cols

# --- D. CONSOLIDATE PATIENT-LEVEL DATA ---
patient_df = pd.concat([patient_vitals_labs, daily_vol_stats, med_stats], axis=1).reset_index()

# Length of Stay
los_series = df.groupby('encounterId')['utcChartTime'].agg(
    lambda x: (x.max() - x.min()).total_seconds() / 86400
)
patient_df = patient_df.merge(los_series.rename('Length_of_Stay_Days'), on='encounterId', how='left')
patient_df['Length_of_Stay_Days'] = patient_df['Length_of_Stay_Days'].replace(0, 1/24)

# Handle categorical strings
for col in ['first_careunit', 'sex_or_gender']:
    if col in patient_df.columns:
        patient_df[col] = patient_df[col].astype(str).replace(['nan', 'None', 'NaN', '<NA>'], 'Unknown')

# ---------------------------------------------------------
# 5. CREATE PREDICTION-LEVEL DATA
# ---------------------------------------------------------
print("Creating prediction-level data...")

if 'y_test' not in locals() or 'y_pred_proba' not in locals():
    raise ValueError("y_test or y_pred_proba not found. Run model predictions first.")

# For external validation, df IS the full dataset
pred_level_df = df.copy()

if len(y_test) == len(df):
    pred_level_df['y_true'] = y_test
    pred_level_df['y_prob'] = y_pred_proba
else:
    raise ValueError(f"Length mismatch: y_test ({len(y_test)}) != df ({len(df)})")

# Handle categorical strings in prediction-level data
for col in ['first_careunit', 'sex_or_gender']:
    if col in pred_level_df.columns:
        pred_level_df[col] = pred_level_df[col].astype(str).replace(['nan', 'None', 'NaN', '<NA>'], 'Unknown')

# ---------------------------------------------------------
# 6. DEFINE ANALYSIS GROUPS
# ---------------------------------------------------------
print("Defining analysis groups...")

# --- PATIENT-LEVEL: Full Dataset only (no TP/TN/FP/FN) ---
patient_groups = {
    'Full_Dataset': patient_df,
}

# --- PREDICTION-LEVEL: TP/TN/FP/FN (each row = one prediction) ---
pred_groups = {
    'All': pred_level_df,
    'TP': pred_level_df[(pred_level_df['y_true'] == 1) & (pred_level_df['y_prob'] >= 0.50)],
    'TN': pred_level_df[(pred_level_df['y_true'] == 0) & (pred_level_df['y_prob'] < 0.50)],
    'FN': pred_level_df[(pred_level_df['y_true'] == 1) & (pred_level_df['y_prob'] < 0.50)],
    'FP': pred_level_df[(pred_level_df['y_true'] == 0) & (pred_level_df['y_prob'] >= 0.50)],
}

# ---------------------------------------------------------
# 7. HELPER FUNCTIONS
# ---------------------------------------------------------

def fmt(v, decimals=2):
    """Format numeric values safely."""
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "-"
        return f"{v:.{decimals}f}"
    except:
        return "-"

def calculate_stats_compact(df_sub, col):
    """Calculate N, Mean±SD, Median (Q1, Q3) for a column."""
    if df_sub.empty or col not in df_sub.columns:
        return {"N": "0", "Mean ± SD": "-", "Median (Q1, Q3)": "-"}
    
    s = df_sub[col].dropna()
    if len(s) == 0:
        return {"N": "0", "Mean ± SD": "-", "Median (Q1, Q3)": "-"}
    
    mean_val = s.mean()
    std_val = s.std()
    median_val = s.median()
    q1_val = s.quantile(0.25)
    q3_val = s.quantile(0.75)
    
    return {
        "N": str(int(len(s))),
        "Mean ± SD": f"{fmt(mean_val)} ± {fmt(std_val)}",
        "Median (Q1, Q3)": f"{fmt(median_val)} ({fmt(q1_val)}, {fmt(q3_val)})"
    }

# ---------------------------------------------------------
# 8. GENERATE TABLES
# ---------------------------------------------------------
print("Generating tables...")

# ==========================================================
# TABLE 1: PATIENT-LEVEL - FULL DATASET ONLY
# ==========================================================

rows_patient_continuous = []
rows_patient_categorical = []

# --- CONTINUOUS VARIABLES (Patient-Level) ---
demo_vars = [('age', 'Age (years)'), ('Length_of_Stay_Days', 'Length of Stay (days)')]
vital_vars = [(f'{v}_mean', f'{v} (mean)') for v in vitals if f'{v}_mean' in patient_df.columns]
lab_vars = [(f'{v}_mean', f'{v} (mean)') for v in labs if f'{v}_mean' in patient_df.columns]
med_vars = [(f'{d}_mean', f'{d} (mean when given)') for d in drugs if f'{d}_mean' in patient_df.columns]
med_count_vars = [(f'{d}_count', f'{d} (# administrations)') for d in drugs if f'{d}_count' in patient_df.columns]
vol_vars = [(f'{v}_mean_daily', f'{v} (daily sum, mean)') for v in volumes if f'{v}_mean_daily' in patient_df.columns]

sections = [
    ("DEMOGRAPHICS", demo_vars),
    ("VITALS", vital_vars),
    ("LABS", lab_vars),
    ("MEDICATIONS (Dose when administered)", med_vars),
    ("MEDICATIONS (# of administrations)", med_count_vars),
    ("VOLUMES (Daily totals)", vol_vars),
]

for section_name, var_list in sections:
    if not var_list:
        continue
    rows_patient_continuous.append({'Variable': f"--- {section_name} ---"})
    for col, label in var_list:
        row = {'Variable': label}
        for g_name, g_df in patient_groups.items():
            stats = calculate_stats_compact(g_df, col)
            for stat_name, stat_val in stats.items():
                row[f'{g_name}_{stat_name}'] = stat_val
        rows_patient_continuous.append(row)

# --- CATEGORICAL VARIABLES (Patient-Level) ---
cat_vars = ['sex_or_gender', 'first_careunit']

for var in cat_vars:
    if var not in patient_df.columns:
        continue
    
    rows_patient_categorical.append({'Variable': f"--- {var} ---"})
    unique_vals = sorted(patient_df[var].astype(str).unique())
    
    for val in unique_vals:
        row = {'Variable': val}
        for g_name, g_df in patient_groups.items():
            if g_df.empty or var not in g_df.columns:
                count, total, pct = 0, 0, 0.0
            else:
                count = (g_df[var].astype(str) == val).sum()
                total = len(g_df)
                pct = (count / total * 100) if total > 0 else 0
            
            row[f'{g_name}_N'] = str(total)
            row[f'{g_name}_n (%)'] = f"{count} ({pct:.1f}%)"
        rows_patient_categorical.append(row)

# ==========================================================
# TABLE 2: PREDICTION-LEVEL - TP/TN/FP/FN
# ==========================================================

rows_pred_continuous = []
rows_pred_categorical = []

# --- CONTINUOUS VARIABLES (Prediction-Level) ---
pred_vitals = [(v, v) for v in vitals if v in pred_level_df.columns]
pred_labs = [(l, l) for l in labs if l in pred_level_df.columns]
pred_drugs = [(d, d) for d in drugs if d in pred_level_df.columns]
pred_volumes = [(v, v) for v in volumes if v in pred_level_df.columns]
pred_meta = [('y_prob', 'Prediction Probability')]

pred_sections = [
    ("PREDICTION", pred_meta),
    ("VITALS (per prediction)", pred_vitals),
    ("LABS (per prediction)", pred_labs),
    ("MEDICATIONS (per prediction)", pred_drugs),
    ("VOLUMES (per prediction)", pred_volumes),
]

for section_name, var_list in pred_sections:
    if not var_list:
        continue
    rows_pred_continuous.append({'Variable': f"--- {section_name} ---"})
    for col, label in var_list:
        row = {'Variable': label}
        for g_name, g_df in pred_groups.items():
            stats = calculate_stats_compact(g_df, col)
            for stat_name, stat_val in stats.items():
                row[f'{g_name}_{stat_name}'] = stat_val
        rows_pred_continuous.append(row)

# --- CATEGORICAL VARIABLES (Prediction-Level) ---
for var in cat_vars:
    if var not in pred_level_df.columns:
        continue
    
    rows_pred_categorical.append({'Variable': f"--- {var} ---"})
    unique_vals = sorted(pred_level_df[var].astype(str).unique())
    
    for val in unique_vals:
        row = {'Variable': val}
        for g_name, g_df in pred_groups.items():
            if g_df.empty or var not in g_df.columns:
                count, total, pct = 0, 0, 0.0
            else:
                count = (g_df[var].astype(str) == val).sum()
                total = len(g_df)
                pct = (count / total * 100) if total > 0 else 0
            
            row[f'{g_name}_N'] = str(total)
            row[f'{g_name}_n (%)'] = f"{count} ({pct:.1f}%)"
        rows_pred_categorical.append(row)

# ---------------------------------------------------------
# 9. ADD SUMMARY ROW WITH COUNTS
# ---------------------------------------------------------

# Patient-level summary
patient_summary = {'Variable': 'TOTAL PATIENTS'}
for g_name, g_df in patient_groups.items():
    patient_summary[f'{g_name}_N'] = str(len(g_df))
rows_patient_continuous.insert(0, patient_summary)

# Prediction-level summary
pred_summary = {'Variable': 'TOTAL PREDICTIONS'}
for g_name, g_df in pred_groups.items():
    pred_summary[f'{g_name}_N'] = str(len(g_df))
rows_pred_continuous.insert(0, pred_summary)

# ---------------------------------------------------------
# 10. SAVE TO EXCEL
# ---------------------------------------------------------
print("Saving to Excel...")

output_path = 'Table_Failure_Analysis_MIMIC_External.xlsx'

with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    # Sheet 1: Patient-Level Continuous
    pd.DataFrame(rows_patient_continuous).to_excel(
        writer, sheet_name='Patient_Continuous', index=False
    )
    
    # Sheet 2: Patient-Level Categorical
    pd.DataFrame(rows_patient_categorical).to_excel(
        writer, sheet_name='Patient_Categorical', index=False
    )
    
    # Sheet 3: Prediction-Level Continuous
    pd.DataFrame(rows_pred_continuous).to_excel(
        writer, sheet_name='Prediction_Continuous', index=False
    )
    
    # Sheet 4: Prediction-Level Categorical
    pd.DataFrame(rows_pred_categorical).to_excel(
        writer, sheet_name='Prediction_Categorical', index=False
    )
    
    # Format columns
    for sheet in writer.sheets.values():
        sheet.set_column(0, 0, 50)  # Variable column
        sheet.set_column(1, 100, 20)  # Data columns (wider for combined stats)

# ---------------------------------------------------------
# 11. PRINT SUMMARY WITH METRICS
# ---------------------------------------------------------
print(f"\n{'='*60}")
print("SUMMARY - MIMIC EXTERNAL VALIDATION")
print('='*60)

print(f"\nPatient-Level:")
for g_name, g_df in patient_groups.items():
    print(f"  {g_name}: {len(g_df)} patients")

print(f"\nPrediction-Level:")
for g_name, g_df in pred_groups.items():
    print(f"  {g_name}: {len(g_df)} predictions")

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

y_true_pred = pred_level_df['y_true'].values
y_prob_pred = pred_level_df['y_prob'].values
y_pred_pred = (y_prob_pred >= 0.5).astype(int)

print(f"\nPrediction-Level Metrics:")
print(f"  Accuracy:  {accuracy_score(y_true_pred, y_pred_pred):.3f}")
print(f"  Precision: {precision_score(y_true_pred, y_pred_pred, zero_division=0):.3f}")
print(f"  Recall:    {recall_score(y_true_pred, y_pred_pred, zero_division=0):.3f}")
print(f"  F1 Score:  {f1_score(y_true_pred, y_pred_pred, zero_division=0):.3f}")
print(f"  AUC-ROC:   {roc_auc_score(y_true_pred, y_prob_pred):.3f}")

print(f"\nOutput saved to: {output_path}")

In [ ]:
volumes = ['fluids_ml', 'blood_input', 'colloids_ml', 'harnk_ml', 'drain_sum']
drugs = ['combined_vaso']
vitals = ['heart_rate', 'respiratory_rate', 'spo2', 'blood_pressure_systolic_mmHg', 
          'blood_pressure_diastolic_mmHg', 'blood_pressure_mean_mmHg']
labs = ['hemoglobin_g/dl', 'lactate_mmol/l', 'fibrinogen_mg/dl', 'platelet_count_G/l', 'base_excess_mmol/l']


In [ ]:
df['fluids_ml'].describe()

In [ ]:
features = volumes+drugs+vitals+labs
for col in features:
    
    print(col)
    print(df_test[col].describe())
    print(df_test.loc[df_test[col]>0,col].describe())
    print('___________________________________ \n')


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Ensure base_dir is defined and exists
base_dir = 'xgb_models_best'
os.makedirs(base_dir, exist_ok=True)

# Number of bootstrap samples
n_bootstraps = 100
rng = np.random.RandomState(seed=42)  # For reproducibility

# Arrays to store bootstrapped FPR and TPR
bootstrapped_fpr = []
bootstrapped_tpr = []

# Convert y_test and X_test to numpy arrays if they aren't already
y_test_np = np.array(y_test)
X_test_np = np.array(X_test)
y_pred_proba_np = np.array(y_pred_proba)

for i in range(n_bootstraps):
    # Sample with replacement from the test set
    indices = rng.randint(0, len(X_test_np), len(X_test_np))
    
    # Ensure both classes are present in the bootstrap sample
    if len(np.unique(y_test_np[indices])) < 2:
        continue
    
    # Extract the sampled true labels and predicted probabilities
    y_test_sample = y_test_np[indices]
    y_pred_proba_sample = y_pred_proba_np[indices]
    
    # Compute ROC curve
    fpr, tpr, _ = roc_curve(y_test_sample, y_pred_proba_sample)
    bootstrapped_fpr.append(fpr)
    bootstrapped_tpr.append(tpr)

# Define mean FPR for interpolation
mean_fpr = np.linspace(0, 1, 100)

# Interpolate TPRs at the mean FPR points
interpolated_tprs = []
for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr):
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0  # Ensure the curve starts at (0,0)
    interpolated_tprs.append(interp_tpr)

# Convert list to array for percentile calculation
interpolated_tprs = np.array(interpolated_tprs)

# Compute mean and confidence intervals
mean_tpr = np.mean(interpolated_tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)
tpr_lower = np.percentile(interpolated_tprs, 5, axis=0)
tpr_upper = np.percentile(interpolated_tprs, 95, axis=0)

auc = auc(fpr, tpr)

mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.array([np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr)])



# Compute the lower and upper bound for tpr
tpr_lower = np.percentile(mean_tpr, 5, axis=0)
tpr_upper = np.percentile(mean_tpr, 95, axis=0)

# Specificity is 1 - FPR
specificity =  1 - fpr

# Set the style and context with seaborn
sns.set_style("ticks")
sns.set_context("notebook")



# Specificity is 1 - FPR
specificity =  1 - fpr





# Plotting
plt.figure(figsize=(8, 6))

plt.plot([1, 0], [0, 1], 'k--')  # Invert x-axis
plt.plot(specificity, tpr, label=f'(AUC = {auc:.2f})')
plt.fill_between(1-mean_fpr, tpr_lower, tpr_upper, alpha=0.2)

plt.xlim([1, 0])  # Invert x-axis limits
plt.ylim([0, 1])
#plt.xticks([1,0.8,0.6,0.4,0.2,0])  # Set x-axis tick locations
#plt.xticks([1,0.8,0.6,0.4,0.2,0])  # Set x-axis tick labels

# Labels and Title
plt.xlabel('Specificity')
plt.ylabel('Sensitivity')
plt.title('ROC curve for hemoglobin prediction', fontsize=16)
# Legend
plt.legend(loc="lower right")

# Ensure equal aspect ratio
plt.gca().set_aspect('equal')

# Tight layout for better spacing
plt.tight_layout()

# Save the ROC curve
roc_curve_path = os.path.join(base_dir, 'roc_curve_bootstrapped_MIMIC.png')
plt.savefig(roc_curve_path)
plt.show()
plt.close()
print(f"Bootstrapped ROC curve saved to {roc_curve_path}")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Ensure base_dir is defined and exists
base_dir = 'xgb_models_best'
os.makedirs(base_dir, exist_ok=True)

# Number of bootstrap samples
n_bootstraps = 100
rng = np.random.RandomState(seed=42)  # For reproducibility

# Arrays to store bootstrapped FPR and TPR
bootstrapped_fpr = []
bootstrapped_tpr = []

# Convert y_test and X_test to numpy arrays if they aren't already
y_test_np = np.array(y_test)
X_test_np = np.array(X_test)
y_pred_proba_np = np.array(y_pred_proba)

for i in range(n_bootstraps):
    # Sample with replacement from the test set
    indices = rng.randint(0, len(X_test_np), len(X_test_np))
    
    # Ensure both classes are present in the bootstrap sample
    if len(np.unique(y_test_np[indices])) < 2:
        continue
    
    # Extract the sampled true labels and predicted probabilities
    y_test_sample = y_test_np[indices]
    y_pred_proba_sample = y_pred_proba_np[indices]
    
    # Compute ROC curve
    fpr, tpr, _ = roc_curve(y_test_sample, y_pred_proba_sample)
    bootstrapped_fpr.append(fpr)
    bootstrapped_tpr.append(tpr)

# Define mean FPR for interpolation
mean_fpr = np.linspace(0, 1, 100)

# Interpolate TPRs at the mean FPR points
interpolated_tprs = []
for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr):
    interp_tpr = np.interp(mean_fpr, fpr, tpr)
    interp_tpr[0] = 0.0  # Ensure the curve starts at (0,0)
    interpolated_tprs.append(interp_tpr)

# Convert list to array for percentile calculation
interpolated_tprs = np.array(interpolated_tprs)

# Compute mean and confidence intervals
mean_tpr = np.mean(interpolated_tprs, axis=0)
mean_auc = auc(mean_fpr, mean_tpr)
tpr_lower = np.percentile(interpolated_tprs, 5, axis=0)
tpr_upper = np.percentile(interpolated_tprs, 95, axis=0)

auc = auc(fpr, tpr)

mean_fpr = np.linspace(0, 1, 100)
mean_tpr = np.array([np.interp(mean_fpr, fpr, tpr) for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr)])



# Compute the lower and upper bound for tpr
tpr_lower = np.percentile(mean_tpr, 5, axis=0)
tpr_upper = np.percentile(mean_tpr, 95, axis=0)

# Specificity is 1 - FPR
specificity =  1 - fpr

# Set the style and context with seaborn
sns.set_style("ticks")
sns.set_context("notebook")



# Specificity is 1 - FPR
specificity =  1 - fpr





# Plotting
plt.figure(figsize=(8, 6))

plt.plot([1, 0], [0, 1], 'k--')  # Invert x-axis
plt.plot(specificity, tpr, label=f'(AUC = {auc:.2f})')
plt.fill_between(1-mean_fpr, tpr_lower, tpr_upper, alpha=0.2)

plt.xlim([1, 0])  # Invert x-axis limits
plt.ylim([0, 1])
#plt.xticks([1,0.8,0.6,0.4,0.2,0])  # Set x-axis tick locations
#plt.xticks([1,0.8,0.6,0.4,0.2,0])  # Set x-axis tick labels

# Labels and Title
plt.xlabel('Specificity')
plt.ylabel('Sensitivity')
plt.title('ROC curve for hemoglobin prediction', fontsize=16)
# Legend
plt.legend(loc="lower right")

# Ensure equal aspect ratio
plt.gca().set_aspect('equal')

# Tight layout for better spacing
plt.tight_layout()

# Save the ROC curve
roc_curve_path = os.path.join(base_dir, 'roc_curve_bootstrapped_MIMIC.png')
plt.savefig(roc_curve_path)
plt.show()
plt.close()
print(f"Bootstrapped ROC curve saved to {roc_curve_path}")

In [ ]:
df_test

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

# Ensure directory exists
base_dir = 'xgb_models_best'
os.makedirs(base_dir, exist_ok=True)

# Bootstrapping setup
n_bootstraps = 100  # Consider increasing to 100 or 1000 for final results
rng = np.random.RandomState(seed=42)

# Convert overall predictions to a 1D numpy array 
# (No need for [:, 1] since it's already 1D!)
y_pred_proba_all = np.array(y_pred_proba)

# Dictionary to handle gender groups, labels, and plotting colors
gender_groups = {
    1.0: {'label': 'Gender: female', 'color': 'red'},
    0.0: {'label': 'Gender: male', 'color': 'blue'}
}

# Set up the plot base
sns.set_style("ticks")
sns.set_context("notebook")
plt.figure(figsize=(8, 6))
plt.plot([1, 0], [0, 1], 'k--', label='Random Chance')  # Diagonal line

# Loop through each gender
for gender_val, style in gender_groups.items():
    
    # 1. Create a mask for the specific gender
    mask = df_test['sex_or_gender'] == gender_val
    
    # 2. Extract true labels and corresponding predictions for this group
    y_test_np = df_test.loc[mask, 'y_binary'].values
    y_pred_proba_np = y_pred_proba_all[mask]  # Fixed: Just use the mask directly
    
    bootstrapped_fpr = []
    bootstrapped_tpr = []
    
    # Run bootstrapping
    for i in range(n_bootstraps):
        indices = rng.randint(0, len(y_test_np), len(y_test_np))
        
        # Ensure both classes are present
        if len(np.unique(y_test_np[indices])) < 2:
            continue
            
        y_test_sample = y_test_np[indices]
        y_pred_proba_sample = y_pred_proba_np[indices]
        
        fpr, tpr, _ = roc_curve(y_test_sample, y_pred_proba_sample)
        bootstrapped_fpr.append(fpr)
        bootstrapped_tpr.append(tpr)

    # Interpolate TPRs at mean FPR points
    mean_fpr = np.linspace(0, 1, 100)
    interpolated_tprs = []
    for fpr, tpr in zip(bootstrapped_fpr, bootstrapped_tpr):
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        interpolated_tprs.append(interp_tpr)

    interpolated_tprs = np.array(interpolated_tprs)
    
    # Compute mean and CI for this gender
    mean_tpr = np.mean(interpolated_tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    
    tpr_lower = np.percentile(interpolated_tprs, 5, axis=0)
    tpr_upper = np.percentile(interpolated_tprs, 95, axis=0)
    
    specificity = 1 - mean_fpr
    
    # Plotting for this specific gender
    plt.plot(specificity, mean_tpr, color=style['color'], 
             label=f"{style['label']} (Mean AUC = {mean_auc:.2f})")
    
    plt.fill_between(specificity, tpr_lower, tpr_upper, color=style['color'], 
                     alpha=0.2, label=f"{style['label']} 90% CI")

# Final plot adjustments
plt.xlim([1, 0])  # Invert x-axis
plt.ylim([0, 1])
plt.xlabel('Specificity')
plt.ylabel('Sensitivity')
plt.title('MIMIC: ROC curve by Gender', fontsize=16)

# Create legend (removing duplicates if any)
handles, labels = plt.gca().get_legend_handles_labels()
plt.legend(handles, labels, loc="lower right", fontsize=10)

plt.gca().set_aspect('equal')
plt.tight_layout()

# Save and show
roc_curve_path = os.path.join(base_dir, 'roc_curve_gender_MIMIC.png')
plt.savefig(roc_curve_path, dpi=300)
plt.show()
plt.close()

print(f"Bootstrapped combined ROC curve saved to {roc_curve_path}")

In [ ]:
base_dir

In [ ]:
df_test.loc[df_test['sex_or_gender']==1, 'y_binary']

In [ ]:
df_test.loc[df_test['sex_or_gender']==1, feature_cols].reset_index()

In [ ]:
df_test['sex_or_gender'].value_counts()